# 05_1d_empymod_inversion

Stochastic 1D layered phase-only inversion with `empymod`.

This notebook:
- loads run data (`Hx_data.rss`, `Hz_data.rss`),
- extracts phase-only observations for `Hx-Hx` and `Hx-Hz` across all selected frequencies,
- inverts a 1D layered isotropic model independently for each Tx using global optimization,
- interpolates the Tx-wise 1D models into a pseudo-2D resistivity section over a homogeneous background,
- exports results to NPZ/JSON and optional RSS.


In [ ]:
from pathlib import Path
import json
import os
import re
import signal
import sys
import traceback

import numpy as np
from IPython.display import display

try:
    import ipywidgets as ipw
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import empymod
    from scipy.optimize import differential_evolution, dual_annealing
except Exception as exc:
    raise RuntimeError(
        "Missing dependencies. Install with: pip install empymod scipy ipywidgets plotly"
    ) from exc

# Project root
ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.modules.fd_visualization import (
    build_trace_index,
    compute_amp_phase_for_fd_outputs,
    load_rss_traces,
)
from scripts.modules.segy import write_resistivity_to_segy_from_template
from third_party.rockseis.io.rsfile import rsfile

RUN_DIR_PATTERN = re.compile(r"^InversionRun(\\d+)$")
INV_INPUT_DIR = ROOT / "InversionInput"
FDMODEL_DIR = ROOT / "FDmodel"
FDMODEL_DATA_DIR = FDMODEL_DIR / "Data"
SG_TRUE_PATH = FDMODEL_DIR / "sg.rss"
SETUP_META = FDMODEL_DIR / "setup_metadata.json"
VOILA_PID_FILE = ROOT / ".voila_1d_empymod_server.pid"

state = {
    "features": None,
    "tx_results": {},
    "section": None,
    "messages": [],
}


def push_message(msg):
    state["messages"].append(str(msg))
    if len(state["messages"]) > 20:
        state["messages"] = state["messages"][-20:]
    status_out.value = "\n".join(state["messages"])


def list_run_dirs(root_dir):
    out = []
    for child in Path(root_dir).iterdir():
        if child.is_dir():
            m = RUN_DIR_PATTERN.match(child.name)
            if m:
                out.append((int(m.group(1)), child))
    out.sort(key=lambda x: x[0])
    return out


def default_frequencies():
    setup_meta = ROOT / "FDmodel" / "setup_metadata.json"
    if setup_meta.exists():
        try:
            meta = json.loads(setup_meta.read_text())
            vals = [float(v) for v in meta.get("flist_hz", [])]
            if vals:
                return vals
        except Exception:
            pass
    return [2e3, 4e3, 6e3]


def parse_freqs(text):
    vals = [float(v.strip()) for v in str(text).split(",") if v.strip()]
    if not vals:
        raise ValueError("Frequency list cannot be empty.")
    arr = np.asarray(vals, dtype=float)
    if np.any(arr <= 0):
        raise ValueError("Frequencies must be positive.")
    return arr


def wrap_phase_rad(phi):
    return np.arctan2(np.sin(phi), np.cos(phi))


def compensated_obs_cos(obs_phase, phase_comp_deg):
    """Cosine-domain observable with systematic phase compensation on observed phase."""
    shift = np.deg2rad(float(phase_comp_deg))
    return np.cos(wrap_phase_rad(np.asarray(obs_phase, dtype=float) - shift))


def conductivity_to_resistivity(grid, min_sigma=1e-12):
    sigma = np.clip(np.asarray(grid, dtype=float), min_sigma, 1e12)
    return 1.0 / sigma


def _read_rss_model(path):
    f = rsfile()
    f.read(str(path))
    data = np.asarray(f.data, dtype=float)
    data = np.squeeze(data)
    if data.ndim != 2:
        raise ValueError(f"Expected 2D model, got shape {data.shape}")
    nx, nz = int(data.shape[0]), int(data.shape[1])
    grid = np.asarray(data.T, dtype=float)
    dx = float(f.geomD[0]) if f.geomD[0] else 1.0
    ox = float(f.geomO[0])
    iz = 2 if (len(f.geomN) > 2 and int(f.geomN[2]) > 0) else 1
    dz = float(f.geomD[iz]) if f.geomD[iz] else 1.0
    oz = float(f.geomO[iz])
    x = ox + dx * np.arange(nx)
    z = oz + dz * np.arange(nz)
    return x, z, grid


def _extract_positions():
    candidates = [
        FDMODEL_DATA_DIR / "Hxshot.rss",
        FDMODEL_DATA_DIR / "Hx_data.rss",
        INV_INPUT_DIR / "Hx_data.rss",
    ]

    hx_path = None
    for c in candidates:
        if c.exists():
            hx_path = c
            break
    if hx_path is None:
        return np.array([]), np.array([]), np.array([]), np.array([])

    try:
        meta = load_rss_traces(hx_path)
        src = np.column_stack((meta["src_x"], meta["src_z"]))
        rec = np.column_stack((meta["rx_x"], meta["rx_z"]))
        src_u = np.unique(np.round(src, 6), axis=0)
        rec_u = np.unique(np.round(rec, 6), axis=0)
        return src_u[:, 0], src_u[:, 1], rec_u[:, 0], rec_u[:, 1]
    except Exception:
        return np.array([]), np.array([]), np.array([]), np.array([])


def get_real_data_paths():
    candidates = [
        (FDMODEL_DATA_DIR / "Hxshot.rss", FDMODEL_DATA_DIR / "Hzshot.rss"),
        (FDMODEL_DATA_DIR / "Hx_data.rss", FDMODEL_DATA_DIR / "Hz_data.rss"),
        (INV_INPUT_DIR / "Hx_data.rss", INV_INPUT_DIR / "Hz_data.rss"),
    ]

    for hx, hz in candidates:
        if hx.exists() and hz.exists():
            return hx, hz

    # Return first pair for clearer error paths upstream.
    return candidates[0]


def _ensure_freq_rx_shape(arr, nfreq, nrx):
    arr = np.asarray(arr)
    if arr.ndim == 0:
        out = np.full((nfreq, nrx), arr, dtype=complex)
        return out
    if arr.ndim == 1:
        if arr.size == nfreq:
            return np.repeat(arr[:, None], nrx, axis=1)
        if arr.size == nrx:
            return np.repeat(arr[None, :], nfreq, axis=0)
    if arr.shape == (nfreq, nrx):
        return arr
    if arr.shape == (nrx, nfreq):
        return arr.T
    if arr.size == nfreq * nrx:
        return arr.reshape((nfreq, nrx))
    raise ValueError(f"Cannot reshape response with shape {arr.shape} to ({nfreq}, {nrx})")


def extract_phase_features(freqs, start_t=None, end_t=None):
    hx_path, hz_path = get_real_data_paths()
    if not hx_path.exists() or not hz_path.exists():
        raise FileNotFoundError("Hx/Hz observation data not found in FDmodel/Data or InversionInput")

    amp_phase = compute_amp_phase_for_fd_outputs(
        hx_path,
        hz_path,
        freqs=freqs,
        start_t=start_t,
        end_t=end_t,
        n_pairs=3,
    )
    geo = amp_phase["geometry"]
    tx_idx = np.asarray(geo["tx_idx_per_trace"], dtype=int)
    tx_unique = np.asarray(geo["tx_unique"], dtype=float)

    phi_hx = np.asarray(amp_phase["Hx"]["phi_mean_rad"], dtype=float)
    phi_hz = np.asarray(amp_phase["Hz"]["phi_mean_rad"], dtype=float)

    tx_data = {}
    for tx_id in np.unique(tx_idx):
        tr = np.where(tx_idx == int(tx_id))[0]
        if tr.size == 0:
            continue

        tx_xy = tx_unique[int(tx_id)]
        rx_x = np.asarray(geo["rx_x"], dtype=float)[tr]
        rx_z = np.asarray(geo["rx_z"], dtype=float)[tr]
        off_x = rx_x - float(tx_xy[0])
        off_z = rx_z - float(tx_xy[1])

        obs_hxh = wrap_phase_rad(phi_hx[:, tr])
        obs_hxhz = wrap_phase_rad(phi_hz[:, tr])

        tx_data[int(tx_id)] = {
            "tx_id": int(tx_id),
            "tx_x": float(tx_xy[0]),
            "tx_z": float(tx_xy[1]),
            "rx_x": rx_x,
            "rx_z": rx_z,
            "off_x": off_x,
            "off_z": off_z,
            "freqs": np.asarray(freqs, dtype=float),
            "obs_hxh": obs_hxh,
            "obs_hxhz": obs_hxhz,
        }

    return {
        "run_dir": str(FDMODEL_DATA_DIR),
        "hx_path": str(hx_path),
        "hz_path": str(hz_path),
        "freqs": np.asarray(freqs, dtype=float),
        "tx_data": tx_data,
        "geometry": geo,
    }


def unpack_model_params(params, n_layers, z_start_rel, z_end_rel):
    p = np.asarray(params, dtype=float)
    lrho = p[:n_layers]
    rho = np.power(10.0, lrho)

    # Thickness parameters are optimized in log10-space.
    lthk = p[n_layers:]
    thk_raw = np.power(10.0, lthk)
    thk_raw = np.clip(thk_raw, 1e-9, np.inf)

    span = float(z_end_rel - z_start_rel)
    if span <= 0:
        raise ValueError("Invalid depth window: z_end_rel must be > z_start_rel")

    if thk_raw.size != max(0, n_layers - 1):
        raise ValueError("Thickness parameter count mismatch.")

    if thk_raw.size > 0:
        thk = thk_raw / np.sum(thk_raw) * span
        depth = float(z_start_rel) + np.cumsum(thk)
    else:
        thk = np.array([], dtype=float)
        depth = np.array([], dtype=float)
    return rho, thk, depth


def _ab_with_z_oriented_source(ab_code):
    ab = int(ab_code)
    src_code = ab // 10
    rec_code = ab % 10
    # Empymod component encoding uses +2 to go x->z within E/H families.
    return (src_code + 2) * 10 + rec_code


def forward_empymod_for_tx(
    params,
    tx_entry,
    n_layers,
    z_start_rel,
    z_end_rel,
    tilt_deg,
    ab_hxh=44,
    ab_hxhz=46,
):
    rho, thk, depth = unpack_model_params(params, n_layers, z_start_rel, z_end_rel)
    freqs = np.asarray(tx_entry["freqs"], dtype=float)
    off_x = np.asarray(tx_entry["off_x"], dtype=float)
    off_z = np.asarray(tx_entry["off_z"], dtype=float)

    nfreq = freqs.size
    nrx = off_x.size

    # dipole() expects src=[x, y, z].
    src = [0.0, 0.0, 0.0]

    common = {
        "src": src,
        "depth": depth,
        "res": rho,
        "freqtime": freqs,
        "verb": 0,
        "htarg": {"pts_per_dec": -1},
    }

    def _forward_ab(ab_code):
        # dipole() vectorization is stable for x-array with scalar z; if z varies,
        # fall back to per-receiver modelling.
        if nrx == 0:
            return np.zeros((nfreq, 0), dtype=complex)

        if np.allclose(off_z, off_z[0], rtol=0.0, atol=1e-9):
            rec_vec = [off_x, 0.0, float(off_z[0])]
            out = empymod.dipole(ab=int(ab_code), rec=rec_vec, **common)
            return _ensure_freq_rx_shape(out, nfreq, nrx)

        out = np.full((nfreq, nrx), np.nan + 1j * np.nan, dtype=complex)
        for ir in range(nrx):
            rec_i = [float(off_x[ir]), 0.0, float(off_z[ir])]
            resp_i = empymod.dipole(ab=int(ab_code), rec=rec_i, **common)
            out[:, ir] = _ensure_freq_rx_shape(resp_i, nfreq, 1)[:, 0]
        return out

    # Base response: x-oriented source component.
    hxh = _forward_ab(ab_hxh)
    hxhz = _forward_ab(ab_hxhz)

    # Apply optional source tilt in x-z plane by mixing x and z source components.
    theta = np.deg2rad(float(tilt_deg))
    if abs(theta) > 0.0:
        ab_hxh_z = _ab_with_z_oriented_source(ab_hxh)
        ab_hxhz_z = _ab_with_z_oriented_source(ab_hxhz)
        hxh_z = _forward_ab(ab_hxh_z)
        hxhz_z = _forward_ab(ab_hxhz_z)
        hxh = np.cos(theta) * hxh + np.sin(theta) * hxh_z
        hxhz = np.cos(theta) * hxhz + np.sin(theta) * hxhz_z

    pred_hxh = wrap_phase_rad(np.angle(hxh))
    pred_hxhz = wrap_phase_rad(np.angle(hxhz))
    return pred_hxh, pred_hxhz


def phase_only_objective(
    params,
    tx_entry,
    n_layers,
    z_start_rel,
    z_end_rel,
    tilt_deg,
    phase_comp_deg,
    w_hxh,
    w_hxhz,
    reg_lambda,
    ab_hxh,
    ab_hxhz,
):
    try:
        pred_hxh, pred_hxhz = forward_empymod_for_tx(
            params,
            tx_entry,
            n_layers=n_layers,
            z_start_rel=z_start_rel,
            z_end_rel=z_end_rel,
            tilt_deg=tilt_deg,
            ab_hxh=ab_hxh,
            ab_hxhz=ab_hxhz,
        )
    except Exception:
        return 1e12

    obs_hxh = np.asarray(tx_entry["obs_hxh"], dtype=float)
    obs_hxhz = np.asarray(tx_entry["obs_hxhz"], dtype=float)

    # Cosine-domain misfit with configurable systematic phase compensation.
    # Example: phase_comp_deg=180 -> -cos(obs), phase_comp_deg=90 -> sin(obs).
    obs_hxh_c = compensated_obs_cos(obs_hxh, phase_comp_deg)
    obs_hxhz_c = compensated_obs_cos(obs_hxhz, phase_comp_deg)
    c1 = np.cos(pred_hxh) - obs_hxh_c
    c2 = np.cos(pred_hxhz) - obs_hxhz_c

    m1 = np.isfinite(c1)
    m2 = np.isfinite(c2)
    if not np.any(m1) and not np.any(m2):
        return 1e12

    mis = 0.0
    if np.any(m1):
        # Exact requested form (per-Tx solve): sum over frequency and Rx.
        mis += float(w_hxh) * float(np.nansum(np.where(m1, c1 ** 2, np.nan)))
    if np.any(m2):
        # Exact requested form (per-Tx solve): sum over frequency and Rx.
        mis += float(w_hxhz) * float(np.nansum(np.where(m2, c2 ** 2, np.nan)))

    if reg_lambda > 0.0 and n_layers > 1:
        lrho = np.asarray(params[:n_layers], dtype=float)
        mis += float(reg_lambda) * float(np.mean(np.diff(lrho) ** 2))

    return mis


def build_bounds(n_layers, log10_rho_min, log10_rho_max, log10_thk_min, log10_thk_max):
    b = [(float(log10_rho_min), float(log10_rho_max)) for _ in range(int(n_layers))]
    for _ in range(int(n_layers) - 1):
        b.append((float(log10_thk_min), float(log10_thk_max)))
    return b


def invert_single_tx(tx_entry, cfg):
    bounds = build_bounds(
        cfg["n_layers"],
        cfg["log10_rho_min"],
        cfg["log10_rho_max"],
        cfg["log10_thk_min"],
        cfg["log10_thk_max"],
    )

    obj = lambda p: phase_only_objective(
        p,
        tx_entry=tx_entry,
        n_layers=cfg["n_layers"],
        z_start_rel=cfg["z_start_rel"],
        z_end_rel=cfg["z_end_rel"],
        tilt_deg=cfg["tilt_deg"],
        phase_comp_deg=cfg["phase_comp_deg"],
        w_hxh=cfg["w_hxh"],
        w_hxhz=cfg["w_hxhz"],
        reg_lambda=cfg["reg_lambda"],
        ab_hxh=cfg["ab_hxh"],
        ab_hxhz=cfg["ab_hxhz"],
    )

    rng_seed = int(cfg["seed"])
    method = cfg["optimizer"]

    if method == "dual_annealing":
        out = dual_annealing(
            obj,
            bounds=bounds,
            maxfun=int(cfg["maxfun"]),
            seed=rng_seed,
        )
    else:
        out = differential_evolution(
            obj,
            bounds=bounds,
            maxiter=int(cfg["maxiter"]),
            popsize=int(cfg["popsize"]),
            seed=rng_seed,
            polish=False,
            workers=1,
            updating="deferred",
        )

    best = np.asarray(out.x, dtype=float)
    best_mis = float(out.fun)
    pred_hxh, pred_hxhz = forward_empymod_for_tx(
        best,
        tx_entry=tx_entry,
        n_layers=cfg["n_layers"],
        z_start_rel=cfg["z_start_rel"],
        z_end_rel=cfg["z_end_rel"],
        tilt_deg=cfg["tilt_deg"],
        ab_hxh=cfg["ab_hxh"],
        ab_hxhz=cfg["ab_hxhz"],
    )
    rho, thk, depth_rel = unpack_model_params(
        best,
        cfg["n_layers"],
        cfg["z_start_rel"],
        cfg["z_end_rel"],
    )

    tx_z = float(tx_entry.get("tx_z", 0.0))
    z_top = tx_z + float(cfg["z_start_rel"])
    z_bottom = tx_z + float(cfg["z_end_rel"])
    depth_abs = tx_z + np.asarray(depth_rel, dtype=float)

    return {
        "success": bool(getattr(out, "success", True)),
        "message": str(getattr(out, "message", "")),
        "misfit": best_mis,
        "params": best,
        "rho": rho,
        "thickness": thk,
        "depth": depth_abs,
        "z_top": float(z_top),
        "z_bottom": float(z_bottom),
        "pred_hxh": pred_hxh,
        "pred_hxhz": pred_hxhz,
    }


def build_z_grid_from_results(tx_results, dz=5.0, zmax=None):
    max_depth = 0.0
    for res in tx_results.values():
        if "depth" in res and np.asarray(res["depth"]).size:
            max_depth = max(max_depth, float(np.max(res["depth"])))
    if zmax is not None:
        max_depth = max(max_depth, float(zmax))
    max_depth = max(max_depth, float(dz))
    nz = int(np.ceil(max_depth / dz)) + 1
    return np.arange(nz, dtype=float) * float(dz)


def profile_from_result(res, z_grid, background_rho):
    rho = np.asarray(res.get("rho", []), dtype=float)
    interfaces = np.asarray(res.get("depth", []), dtype=float)
    z_top = float(res.get("z_top", np.min(z_grid) if z_grid.size else 0.0))
    z_bottom = float(res.get("z_bottom", np.max(z_grid) if z_grid.size else z_top))

    out = np.full_like(z_grid, float(background_rho), dtype=float)
    if rho.size == 0:
        return out

    z0 = z_top
    for i in range(rho.size):
        if i < interfaces.size:
            z1 = interfaces[i]
            m = (z_grid >= z0) & (z_grid < z1)
        else:
            z1 = z_bottom
            m = (z_grid >= z0) & (z_grid <= z1)
        out[m] = rho[i]
        z0 = z1
    return out


def interpolate_tx_profiles(tx_results, x_grid, z_grid, background_rho):
    if not tx_results:
        return np.full((z_grid.size, x_grid.size), float(background_rho), dtype=float)

    xs = []
    profiles = []
    ordered_ids = sorted(tx_results.keys())
    for tx_id in ordered_ids:
        res = tx_results[tx_id]
        xs.append(float(res["tx_x"]))
        profiles.append(profile_from_result(res, z_grid, background_rho=background_rho))

    xs = np.asarray(xs, dtype=float)
    profiles = np.asarray(profiles, dtype=float)
    order = np.argsort(xs)
    xs = xs[order]
    profiles = profiles[order, :]

    sec = np.full((z_grid.size, x_grid.size), float(background_rho), dtype=float)

    # Single-Tx case: fill an aperture around Tx instead of a single x sample.
    if xs.size == 1:
        tx_res = tx_results[ordered_ids[0]]
        rx_x = np.asarray(tx_res.get("rx_x", []), dtype=float)
        if rx_x.size > 0 and np.any(np.isfinite(rx_x)):
            x_left = float(np.nanmin(rx_x))
            x_right = float(np.nanmax(rx_x))
            m = (x_grid >= x_left) & (x_grid <= x_right)
        else:
            m = np.zeros_like(x_grid, dtype=bool)

        # If aperture is too narrow relative to grid, assign nearest x cell.
        if not np.any(m):
            ix = int(np.argmin(np.abs(x_grid - xs[0])))
            m[ix] = True

        sec[:, m] = profiles[0, :][:, None]
        return sec

    for iz in range(z_grid.size):
        sec[iz, :] = np.interp(
            x_grid,
            xs,
            profiles[:, iz],
            left=float(background_rho),
            right=float(background_rho),
        )
    return sec


def try_default_x_grid(run_dir):
    run_dir = Path(run_dir)
    candidates = [
        run_dir / "sg0.rss",
        run_dir / "sg_ls.rss",
        SG_TRUE_PATH,
    ]
    for p in candidates:
        if p.exists():
            try:
                x, _, _ = _read_rss_model(p)
                return x
            except Exception:
                pass
    return None


def get_default_model_axes():
    candidates = [
        SG_TRUE_PATH,
        FDMODEL_DIR / "sg0.rss",
        FDMODEL_DIR / "sg_ls.rss",
    ]
    for p in candidates:
        if p.exists():
            try:
                x, z, _ = _read_rss_model(p)
                return np.asarray(x, dtype=float), np.asarray(z, dtype=float)
            except Exception:
                pass
    return None, None


def default_depth_window():
    _, z = get_default_model_axes()
    if z is None or z.size == 0:
        return -200.0, 200.0
    return float(np.nanmin(z)), float(np.nanmax(z))


def write_rho_section_to_rss(template_path, out_path, rho_section):
    template_path = Path(template_path)
    out_path = Path(out_path)
    if not template_path.exists():
        raise FileNotFoundError(f"Template RSS not found: {template_path}")

    f = rsfile()
    f.read(str(template_path))
    data_t = np.asarray(f.data)
    data_t = np.squeeze(data_t)
    nx_t, nz_t = int(data_t.shape[0]), int(data_t.shape[1])

    rho = np.asarray(rho_section, dtype=float)
    if rho.shape != (nz_t, nx_t):
        raise ValueError(
            f"Section shape mismatch. Expected (nz={nz_t}, nx={nx_t}), got {rho.shape}"
        )

    sigma = np.clip(1.0 / np.maximum(rho, 1e-9), 1e-12, 1e6)
    f.data = np.asfortranarray(sigma.T.astype(data_t.dtype, copy=False))
    out_path.parent.mkdir(parents=True, exist_ok=True)
    f.write(str(out_path))


def _resolve_segy_template_path():
    if SETUP_META.exists():
        try:
            meta = json.loads(SETUP_META.read_text())
            p_raw = meta.get("segy_template_path")
            if p_raw:
                p = Path(p_raw).expanduser()
                if not p.is_absolute():
                    p = (ROOT / p).resolve()
                if p.exists():
                    return p, "setup_metadata.json"
                push_message(f"SEG-Y template listed in setup metadata does not exist: {p}")
        except Exception as exc:
            push_message(f"Failed reading setup metadata for SEG-Y template: {exc}")

    fallback = ROOT / "input.segy"
    if fallback.exists():
        return fallback, "fallback ROOT/input.segy"
    return None, None


# -------------------- UI --------------------
def_z_start, def_z_end = default_depth_window()

freqs_input = ipw.Text(
    value=",".join([f"{v:g}" for v in default_frequencies()]),
    description="Freqs (Hz):",
    layout=ipw.Layout(width="360px"),
)
start_time_input = ipw.Text(value="", description="start_t (s)", layout=ipw.Layout(width="240px"))
end_time_input = ipw.Text(value="", description="end_t (s)", layout=ipw.Layout(width="240px"))

n_layers_input = ipw.BoundedIntText(value=5, min=2, max=20, description="n_layers", layout=ipw.Layout(width="190px"))
z_start_input = ipw.FloatText(value=def_z_start, description="z_start", layout=ipw.Layout(width="210px"))
z_end_input = ipw.FloatText(value=def_z_end, description="z_end", layout=ipw.Layout(width="210px"))
tilt_input = ipw.FloatText(value=5.0, description="tilt (deg)", layout=ipw.Layout(width="180px"))
phase_comp_input = ipw.FloatText(value=90.0, description="phi_comp (deg)", layout=ipw.Layout(width="210px"))
background_rho_input = ipw.FloatText(value=10.0, description="rho_bg", layout=ipw.Layout(width="190px"))

rho_min_input = ipw.FloatText(value=0.5, description="rho_min", layout=ipw.Layout(width="180px"))
rho_max_input = ipw.FloatText(value=300.0, description="rho_max", layout=ipw.Layout(width="180px"))
thk_min_input = ipw.FloatText(value=20.0, description="thk_min", layout=ipw.Layout(width="180px"))
thk_max_input = ipw.FloatText(value=400.0, description="thk_max", layout=ipw.Layout(width="180px"))

optimizer_input = ipw.Dropdown(
    options=[("Differential Evolution", "differential_evolution"), ("Dual Annealing", "dual_annealing")],
    value="differential_evolution",
    description="optimizer",
    layout=ipw.Layout(width="310px"),
)
maxiter_input = ipw.BoundedIntText(value=30, min=1, max=2000, description="maxiter", layout=ipw.Layout(width="170px"))
popsize_input = ipw.BoundedIntText(value=10, min=2, max=100, description="popsize", layout=ipw.Layout(width="170px"))
maxfun_input = ipw.BoundedIntText(value=800, min=20, max=100000, description="maxfun", layout=ipw.Layout(width="190px"))
seed_input = ipw.BoundedIntText(value=42, min=0, max=10_000_000, description="seed", layout=ipw.Layout(width="170px"))

w_hxh_input = ipw.FloatText(value=1.0, description="w_hxh", layout=ipw.Layout(width="170px"))
w_hxhz_input = ipw.FloatText(value=1.0, description="w_hxhz", layout=ipw.Layout(width="170px"))
reg_lambda_input = ipw.FloatText(value=0.01, description="reg", layout=ipw.Layout(width="170px"))

ab_hxh_input = ipw.BoundedIntText(value=44, min=11, max=66, description="AB HxHx", layout=ipw.Layout(width="180px"))
ab_hxhz_input = ipw.BoundedIntText(value=46, min=11, max=66, description="AB HxHz", layout=ipw.Layout(width="180px"))

x_step_input = ipw.FloatText(value=10.0, description="dx_out", layout=ipw.Layout(width="170px"))
z_step_input = ipw.FloatText(value=5.0, description="dz_out", layout=ipw.Layout(width="170px"))
zmax_input = ipw.FloatText(value=0.0, description="zmax (0=auto)", layout=ipw.Layout(width="220px"))

load_features_btn = ipw.Button(description="Reload phase features", button_style="primary")
run_inversion_btn = ipw.Button(description="Run all Tx inversions", button_style="warning")
build_section_btn = ipw.Button(description="Build pseudo-2D section", button_style="success")
export_btn = ipw.Button(description="Export results")
quit_btn = ipw.Button(description="Quit GUI server", button_style="danger", layout=ipw.Layout(width="170px"))

qc_tx_select = ipw.Dropdown(options=[("n/a", -1)], value=-1, description="QC Tx")
qc_freq_select = ipw.Dropdown(options=[("n/a", 0)], value=0, description="QC freq")
refresh_qc_btn = ipw.Button(description="Refresh QC plots")

section_out = ipw.Output(layout=ipw.Layout(width="100%", height="640px"))
qc_out = ipw.Output(layout=ipw.Layout(width="100%", height="760px"))
status_out = ipw.Textarea(value="Ready.", description="Status", layout=ipw.Layout(width="100%", height="180px"))


def current_cfg():
    rho_min = float(rho_min_input.value)
    rho_max = float(rho_max_input.value)
    if rho_min <= 0 or rho_max <= 0 or rho_max <= rho_min:
        raise ValueError("Resistivity bounds must satisfy 0 < rho_min < rho_max")
    thk_min = float(thk_min_input.value)
    thk_max = float(thk_max_input.value)
    if thk_min <= 0 or thk_max <= thk_min:
        raise ValueError("Thickness bounds must satisfy 0 < thk_min < thk_max")

    z_start = float(z_start_input.value)
    z_end = float(z_end_input.value)
    if z_end <= z_start:
        raise ValueError("Depth bounds must satisfy z_end > z_start")
    halfspan = 0.5 * (z_end - z_start)

    return {
        "n_layers": int(n_layers_input.value),
        "tilt_deg": float(tilt_input.value),
        "phase_comp_deg": float(phase_comp_input.value),
        "background_rho": float(background_rho_input.value),
        "z_start": z_start,
        "z_end": z_end,
        "z_start_rel": -halfspan,
        "z_end_rel": +halfspan,
        "log10_rho_min": float(np.log10(rho_min)),
        "log10_rho_max": float(np.log10(rho_max)),
        "thk_min": thk_min,
        "thk_max": thk_max,
        "log10_thk_min": float(np.log10(thk_min)),
        "log10_thk_max": float(np.log10(thk_max)),
        "optimizer": str(optimizer_input.value),
        "maxiter": int(maxiter_input.value),
        "popsize": int(popsize_input.value),
        "maxfun": int(maxfun_input.value),
        "seed": int(seed_input.value),
        "w_hxh": float(w_hxh_input.value),
        "w_hxhz": float(w_hxhz_input.value),
        "reg_lambda": float(reg_lambda_input.value),
        "ab_hxh": int(ab_hxh_input.value),
        "ab_hxhz": int(ab_hxhz_input.value),
        "x_step": float(x_step_input.value),
        "z_step": float(z_step_input.value),
        "zmax": float(zmax_input.value),
    }


def on_load_features(_):
    try:
        freqs = parse_freqs(freqs_input.value)
        start_t = None if not start_time_input.value.strip() else float(start_time_input.value)
        end_t = None if not end_time_input.value.strip() else float(end_time_input.value)

        feats = extract_phase_features(freqs=freqs, start_t=start_t, end_t=end_t)
        state["features"] = feats
        state["tx_results"] = {}
        state["section"] = None

        tx_ids = sorted(feats["tx_data"].keys())
        qc_tx_select.options = [(f"Tx {i}", i) for i in tx_ids] if tx_ids else [("n/a", -1)]
        qc_tx_select.value = tx_ids[0] if tx_ids else -1
        qc_freq_select.options = [(f"{f:g} Hz", i) for i, f in enumerate(freqs)] if len(freqs) else [("n/a", 0)]
        qc_freq_select.value = 0

        push_message(
            f"Loaded phase features from FDmodel/Data: {len(tx_ids)} Tx, {len(freqs)} freq(s)."
        )
        push_message("Features include Hx-Hx and Hx-Hz phases (phase-only inversion).")
        return True
    except Exception as exc:
        state["features"] = None
        push_message(f"Load features failed: {exc}")
        push_message(traceback.format_exc())
        return False


def on_run_inversion(_):
    try:
        push_message("Loading phase features from FDmodel/Data...")
        ok = on_load_features(None)
        if not ok:
            raise RuntimeError("Could not load phase features automatically.")
        feats = state.get("features")
        if feats is None:
            raise RuntimeError("Load phase features first.")

        cfg = current_cfg()
        tx_ids = sorted(feats["tx_data"].keys())
        if not tx_ids:
            raise RuntimeError("No Tx groups found in data.")

        results = {}
        push_message(f"Starting stochastic inversion for {len(tx_ids)} Tx...")
        for i, tx_id in enumerate(tx_ids, start=1):
            tx_entry = feats["tx_data"][tx_id]
            push_message(f"Inverting Tx {tx_id} ({i}/{len(tx_ids)})...")
            out = invert_single_tx(tx_entry, cfg)
            out["tx_id"] = int(tx_id)
            out["tx_x"] = float(tx_entry["tx_x"])
            out["tx_z"] = float(tx_entry["tx_z"])
            out["freqs"] = np.asarray(tx_entry["freqs"], dtype=float)
            out["obs_hxh"] = np.asarray(tx_entry["obs_hxh"], dtype=float)
            out["obs_hxhz"] = np.asarray(tx_entry["obs_hxhz"], dtype=float)
            out["rx_x"] = np.asarray(tx_entry["rx_x"], dtype=float)
            out["rx_z"] = np.asarray(tx_entry["rx_z"], dtype=float)
            results[int(tx_id)] = out
            push_message(f"Tx {tx_id} done. Misfit={out['misfit']:.4e}")

        state["tx_results"] = results
        push_message("All Tx inversions completed.")
    except Exception as exc:
        push_message(f"Run inversion failed: {exc}")
        push_message(traceback.format_exc())


def on_build_section(_):
    try:
        tx_results = state.get("tx_results") or {}
        if not tx_results:
            raise RuntimeError("Run Tx inversions first.")

        cfg = current_cfg()
        x_default, z_default = get_default_model_axes()
        if x_default is not None and z_default is not None:
            x_model = np.asarray(x_default, dtype=float)
            z_model = np.asarray(z_default, dtype=float)
        else:
            x_model = try_default_x_grid(FDMODEL_DIR)
            if x_model is None:
                xs = np.array([float(v["tx_x"]) for v in tx_results.values()], dtype=float)
                xs.sort()
                dx = max(1.0, float(cfg["x_step"]))
                x_model = np.arange(xs.min(), xs.max() + dx, dx)

            dz = max(0.1, float(cfg["z_step"]))
            zmax = None if cfg["zmax"] <= 0 else float(cfg["zmax"])
            z_model = build_z_grid_from_results(tx_results, dz=dz, zmax=zmax)

        rho_sec = interpolate_tx_profiles(
            tx_results,
            x_grid=np.asarray(x_model, dtype=float),
            z_grid=np.asarray(z_model, dtype=float),
            background_rho=float(cfg["background_rho"]),
        )

        tx_x, tx_z, rx_x, rx_z = _extract_positions()

        state["section"] = {
            "x": np.asarray(x_model, dtype=float),
            "z": np.asarray(z_model, dtype=float),
            "rho": np.asarray(rho_sec, dtype=float),
            "background_rho": float(cfg["background_rho"]),
            "tx_x": tx_x,
            "tx_z": tx_z,
            "rx_x": rx_x,
            "rx_z": rx_z,
        }

        rho_true = None
        x_true = None
        z_true = None
        if SG_TRUE_PATH.exists():
            try:
                x_true, z_true, g_true = _read_rss_model(SG_TRUE_PATH)
                rho_true = conductivity_to_resistivity(g_true)
            except Exception as exc:
                push_message(f"True-model load warning: {exc}")

        if rho_true is not None:
            finite_vals = [
                np.asarray(rho_sec, dtype=float),
                np.asarray(rho_true, dtype=float),
            ]
            finite = np.concatenate([v[np.isfinite(v)] for v in finite_vals if v.size])
            zmin = float(np.nanmin(finite)) if finite.size else None
            zmax = float(np.nanmax(finite)) if finite.size else None

            fig = make_subplots(
                rows=1,
                cols=2,
                subplot_titles=["Pseudo-2D from 1D inversion", "True model (FDmodel/sg.rss)"],
                horizontal_spacing=0.08,
                shared_xaxes="all",
                shared_yaxes="all",
            )
            fig.add_trace(
                go.Heatmap(
                    x=x_model,
                    y=z_model,
                    z=rho_sec,
                    colorscale="Viridis",
                    zmin=zmin,
                    zmax=zmax,
                    showscale=False,
                ),
                row=1,
                col=1,
            )
            fig.add_trace(
                go.Heatmap(
                    x=x_true,
                    y=z_true,
                    z=rho_true,
                    colorscale="Viridis",
                    zmin=zmin,
                    zmax=zmax,
                    showscale=True,
                    colorbar=dict(title="Ohm.m", x=1.02, len=0.92),
                ),
                row=1,
                col=2,
            )

            for col in (1, 2):
                if tx_x.size:
                    fig.add_trace(
                        go.Scatter(
                            x=tx_x,
                            y=tx_z,
                            mode="markers",
                            marker=dict(symbol="triangle-up", size=7, color="red"),
                            name="TX",
                            showlegend=(col == 1),
                        ),
                        row=1,
                        col=col,
                    )
                if rx_x.size:
                    fig.add_trace(
                        go.Scatter(
                            x=rx_x,
                            y=rx_z,
                            mode="markers",
                            marker=dict(symbol="circle", size=6, color="cyan"),
                            name="RX",
                            showlegend=(col == 1),
                        ),
                        row=1,
                        col=col,
                    )

            fig.update_xaxes(title_text="x (m)", row=1, col=1)
            fig.update_xaxes(title_text="x (m)", row=1, col=2)
            fig.update_yaxes(title_text="z (m)", autorange="reversed", row=1, col=1)
            fig.update_yaxes(title_text="z (m)", autorange="reversed", row=1, col=2)
            fig.update_xaxes(matches="x")
            fig.update_yaxes(matches="y")
            fig.update_layout(
                title="Pseudo-2D section and true model",
                height=620,
                margin=dict(t=70, b=50, l=60, r=100),
                legend=dict(orientation="h", y=-0.12),
            )
        else:
            fig = go.Figure()
            fig.add_trace(
                go.Heatmap(
                    x=x_model,
                    y=z_model,
                    z=rho_sec,
                    colorscale="Viridis",
                    colorbar=dict(title="Ohm.m"),
                )
            )
            if tx_x.size:
                fig.add_trace(
                    go.Scatter(
                        x=tx_x,
                        y=tx_z,
                        mode="markers",
                        marker=dict(symbol="triangle-up", size=7, color="red"),
                        name="TX",
                    )
                )
            if rx_x.size:
                fig.add_trace(
                    go.Scatter(
                        x=rx_x,
                        y=rx_z,
                        mode="markers",
                        marker=dict(symbol="circle", size=6, color="cyan"),
                        name="RX",
                    )
                )
            fig.update_layout(
                title="Pseudo-2D section from per-Tx 1D phase-only inversion",
                xaxis_title="x (m)",
                yaxis_title="z (m)",
                yaxis_autorange="reversed",
                height=560,
                margin=dict(t=60, b=50, l=60, r=20),
                legend=dict(orientation="h", y=-0.15),
            )

        with section_out:
            section_out.clear_output(wait=True)
            display(fig)

        push_message("Pseudo-2D section built and plotted.")
    except Exception as exc:
        push_message(f"Build section failed: {exc}")
        push_message(traceback.format_exc())


def on_refresh_qc(_):
    with qc_out:
        qc_out.clear_output(wait=True)
        try:
            tx_id = int(qc_tx_select.value)
            if tx_id < 0:
                display(ipw.HTML("No Tx selected."))
                return
            res = (state.get("tx_results") or {}).get(tx_id)
            if res is None:
                display(ipw.HTML("Run inversion first to populate QC for this Tx."))
                return

            freqs = np.asarray(res["freqs"], dtype=float)
            obs_hxh = np.asarray(res["obs_hxh"], dtype=float)
            obs_hxhz = np.asarray(res["obs_hxhz"], dtype=float)
            pred_hxh = np.asarray(res["pred_hxh"], dtype=float)
            pred_hxhz = np.asarray(res["pred_hxhz"], dtype=float)

            if freqs.size == 0 or obs_hxh.ndim != 2 or obs_hxhz.ndim != 2:
                display(ipw.HTML("No valid QC data available."))
                return

            fidx = int(qc_freq_select.value) if qc_freq_select.value is not None else 0
            fidx = max(0, min(fidx, freqs.size - 1))

            # Show the exact fitted quantities at selected frequency, per Rx.
            phase_comp_deg = float(current_cfg()["phase_comp_deg"])
            obs_hxh_c = compensated_obs_cos(obs_hxh[fidx, :], phase_comp_deg)
            pred_hxh_c = np.cos(pred_hxh[fidx, :])
            obs_hxhz_c = compensated_obs_cos(obs_hxhz[fidx, :], phase_comp_deg)
            pred_hxhz_c = np.cos(pred_hxhz[fidx, :])

            x_idx_hxh = np.arange(obs_hxh_c.size)
            x_idx_hxhz = np.arange(obs_hxhz_c.size)

            fig = make_subplots(
                rows=1,
                cols=2,
                subplot_titles=[
                    f"Hx-Hx cosine vs local rx (f={freqs[fidx]:g} Hz)",
                    f"Hx-Hz cosine vs local rx (f={freqs[fidx]:g} Hz)",
                ],
                horizontal_spacing=0.10,
            )

            fig.add_trace(
                go.Scatter(x=x_idx_hxh, y=obs_hxh_c, mode="lines+markers", name=f"Obs (comp {phase_comp_deg:g}°)", marker=dict(size=6)),
                row=1,
                col=1,
            )
            fig.add_trace(
                go.Scatter(x=x_idx_hxh, y=pred_hxh_c, mode="lines+markers", name="Pred", marker=dict(size=6)),
                row=1,
                col=1,
            )
            fig.add_trace(
                go.Scatter(x=x_idx_hxhz, y=obs_hxhz_c, mode="lines+markers", name=f"Obs (comp {phase_comp_deg:g}°)", showlegend=False, marker=dict(size=6)),
                row=1,
                col=2,
            )
            fig.add_trace(
                go.Scatter(x=x_idx_hxhz, y=pred_hxhz_c, mode="lines+markers", name="Pred", showlegend=False, marker=dict(size=6)),
                row=1,
                col=2,
            )

            fig.update_xaxes(title_text="Local rx index", row=1, col=1)
            fig.update_xaxes(title_text="Local rx index", row=1, col=2)
            fig.update_yaxes(title_text="cos(phase)", range=[-1.05, 1.05], row=1, col=1)
            fig.update_yaxes(title_text="cos(phase)", range=[-1.05, 1.05], row=1, col=2)
            fig.update_layout(
                title=f"Tx {tx_id} cosine-domain QC (phi_comp={phase_comp_deg:g} deg)",
                height=420,
                margin=dict(t=50, b=45, l=60, r=20),
                legend=dict(orientation="h", y=-0.18),
            )
            display(fig)

            tx_ids = sorted((state.get("tx_results") or {}).keys())
            if tx_ids:
                mis = [float(state["tx_results"][k]["misfit"]) for k in tx_ids]
                tx_x = [float(state["tx_results"][k]["tx_x"]) for k in tx_ids]
                fig2 = go.Figure()
                fig2.add_trace(go.Scatter(x=tx_x, y=mis, mode="lines+markers", name="Misfit"))
                fig2.update_layout(
                    title="Misfit vs Tx position",
                    xaxis_title="Tx x (m)",
                    yaxis_title="Objective",
                    height=260,
                    margin=dict(t=40, b=40, l=60, r=20),
                )
                display(fig2)

            display(ipw.HTML(f"<b>Tx {tx_id}</b> best misfit: {res['misfit']:.4e}"))
        except Exception as exc:
            display(ipw.HTML(f"QC plot failed: {exc}"))


def on_export(_):
    try:
        run_path = FDMODEL_DIR
        out_dir = run_path / "OneDInversion"
        out_dir.mkdir(parents=True, exist_ok=True)

        tx_results = state.get("tx_results") or {}
        section = state.get("section")
        if not tx_results:
            raise RuntimeError("No inversion results to export.")

        tx_ids = sorted(tx_results.keys())
        max_layers = max(int(tx_results[k]["rho"].size) for k in tx_ids)
        max_thk = max(int(tx_results[k]["thickness"].size) for k in tx_ids)

        tx_x = np.array([tx_results[k]["tx_x"] for k in tx_ids], dtype=float)
        tx_z = np.array([tx_results[k]["tx_z"] for k in tx_ids], dtype=float)
        misfit = np.array([tx_results[k]["misfit"] for k in tx_ids], dtype=float)

        rho_arr = np.full((len(tx_ids), max_layers), np.nan, dtype=float)
        thk_arr = np.full((len(tx_ids), max_thk), np.nan, dtype=float)
        for i, k in enumerate(tx_ids):
            r = np.asarray(tx_results[k]["rho"], dtype=float)
            t = np.asarray(tx_results[k]["thickness"], dtype=float)
            rho_arr[i, : r.size] = r
            thk_arr[i, : t.size] = t

        npz_path = out_dir / "empymod_1d_phase_inversion_results.npz"
        if section is not None:
            np.savez(
                npz_path,
                tx_ids=np.asarray(tx_ids, dtype=int),
                tx_x=tx_x,
                tx_z=tx_z,
                misfit=misfit,
                rho_layers=rho_arr,
                thickness_layers=thk_arr,
                section_x=np.asarray(section["x"], dtype=float),
                section_z=np.asarray(section["z"], dtype=float),
                section_rho=np.asarray(section["rho"], dtype=float),
            )
        else:
            np.savez(
                npz_path,
                tx_ids=np.asarray(tx_ids, dtype=int),
                tx_x=tx_x,
                tx_z=tx_z,
                misfit=misfit,
                rho_layers=rho_arr,
                thickness_layers=thk_arr,
            )

        summary = {
            "run_dir": str(run_path),
            "n_tx": int(len(tx_ids)),
            "tx_ids": [int(v) for v in tx_ids],
            "misfit": [float(v) for v in misfit],
            "optimizer": str(optimizer_input.value),
            "n_layers": int(n_layers_input.value),
            "tilt_deg": float(tilt_input.value),
            "phase_comp_deg": float(phase_comp_input.value),
            "ab_hxh": int(ab_hxh_input.value),
            "ab_hxhz": int(ab_hxhz_input.value),
            "freqs_hz": [float(v) for v in parse_freqs(freqs_input.value)],
        }
        json_path = out_dir / "empymod_1d_phase_inversion_summary.json"
        json_path.write_text(json.dumps(summary, indent=2))

        section = state.get("section")
        if section is not None:
            template = run_path / "sg0.rss"
            if not template.exists() and SG_TRUE_PATH.exists():
                template = SG_TRUE_PATH
            if template.exists():
                try:
                    rss_path = out_dir / "empymod_1d_pseudo2d_sigma.rss"
                    write_rho_section_to_rss(template, rss_path, section["rho"])
                    push_message(f"Optional RSS exported: {rss_path}")
                except Exception as exc:
                    push_message(f"RSS export skipped: {exc}")

            template_segy, template_src = _resolve_segy_template_path()
            if template_segy is not None:
                try:
                    segy_path = out_dir / "empymod_1d_pseudo2d.segy"
                    info = write_resistivity_to_segy_from_template(
                        template_segy_path=template_segy,
                        output_segy_path=segy_path,
                        resistivity_grid=np.asarray(section["rho"], dtype=float),
                        x_model=np.asarray(section["x"], dtype=float),
                        z_model=np.asarray(section["z"], dtype=float),
                        method="linear",
                    )
                    push_message(f"SEGY export SUCCESS ({template_src}): {segy_path}")
                    if info.get("interpolated"):
                        push_message("SEGY export used interpolation to match template geometry.")
                    else:
                        push_message("SEGY export grid already matched template geometry.")
                except Exception as exc:
                    push_message(f"SEGY export failed: {exc}")
            else:
                push_message("SEGY export skipped: no SEG-Y template found (setup metadata or ROOT/input.segy).")

        push_message(f"Export complete: {npz_path}")
        push_message(f"Summary JSON: {json_path}")
    except Exception as exc:
        push_message(f"Export failed: {exc}")
        push_message(traceback.format_exc())


def on_quit_gui(_):
    push_message("Shutting down 1D inversion GUI server...")

    try:
        if VOILA_PID_FILE.exists():
            pid_text = VOILA_PID_FILE.read_text().strip()
            if pid_text:
                pid = int(pid_text)
                if pid != os.getpid():
                    try:
                        os.kill(pid, signal.SIGTERM)
                        push_message(f"Sent SIGTERM to Voila server PID {pid}.")
                    except Exception as exc:
                        push_message(f"Could not signal Voila PID {pid}: {exc}")
    except Exception as exc:
        push_message(f"Quit warning: {exc}")

    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip and getattr(ip, "kernel", None):
            ip.kernel.do_shutdown(restart=False)
    except Exception as exc:
        push_message(f"Kernel shutdown warning: {exc}")

    try:
        os.kill(os.getpid(), signal.SIGTERM)
    except Exception:
        pass


load_features_btn.on_click(on_load_features)
run_inversion_btn.on_click(on_run_inversion)
build_section_btn.on_click(on_build_section)
refresh_qc_btn.on_click(on_refresh_qc)
export_btn.on_click(on_export)
quit_btn.on_click(on_quit_gui)

app = ipw.VBox([
    ipw.HTML("<h2>05_1d_empymod_inversion</h2>"),
    ipw.HBox([quit_btn], layout=ipw.Layout(justify_content="flex-end")),
    ipw.HTML("<p>Phase-only stochastic 1D inversion per Tx using empymod; pseudo-2D interpolation over background.</p>"),
    ipw.HTML("<h3>1) 1D model parameterization and inversion settings</h3>"),
    ipw.HBox([freqs_input, start_time_input, end_time_input, load_features_btn]),
    ipw.HBox([n_layers_input, z_start_input, z_end_input, tilt_input, phase_comp_input, background_rho_input]),
    ipw.HBox([rho_min_input, rho_max_input, thk_min_input, thk_max_input]),
    ipw.HBox([optimizer_input, maxiter_input, popsize_input, maxfun_input, seed_input]),
    ipw.HBox([w_hxh_input, w_hxhz_input, reg_lambda_input, ab_hxh_input, ab_hxhz_input]),
    ipw.HBox([run_inversion_btn]),
    ipw.HTML("<h3>2) Build pseudo-2D section</h3>"),
    ipw.HBox([build_section_btn]),
    section_out,
    ipw.HTML("<h3>3) QC and export</h3>"),
    ipw.HBox([qc_tx_select, qc_freq_select, refresh_qc_btn, export_btn]),
    qc_out,
    status_out,
])

display(app)
push_message("Ready. Observed phases are always loaded from FDmodel/Data (InversionInput fallback only if needed).")
push_message("Cosine-domain objective uses configurable phase compensation phi_comp (deg).")
on_load_features(None)
